# GMM Tau-p Inversion

Full-Newton Levenberg-Marquardt inversion of a 5-layer ocean-crust model
in the tau-p domain, using the **Global Matrix Method** (Block-Riccati O(N)
solver) as the forward model.

This notebook loads a YAML configuration, runs the inversion, and displays
convergence curves, depth profiles, and trace comparisons inline.

In [ ]:
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# Ensure the project root is on sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Load Configuration

In [ ]:
from Kennett_Reflectivity.inversion_config import load_config

cfg = load_config(Path("configs/default_inversion.yaml"))

# Print model summary
print(f"{'Layer':<15} {'Vp':>6} {'Vs':>6} {'rho':>6} {'h':>8}")
print("-" * 45)
for i in range(cfg.true_model.n_layers):
    name = cfg.layer_names[i]
    vp = cfg.true_model.alpha[i]
    vs = cfg.true_model.beta[i]
    rho = cfg.true_model.rho[i]
    h = cfg.true_model.thickness[i]
    h_str = "inf" if np.isinf(h) else f"{h:.2f}"
    print(f"{name:<15} {vp:6.2f} {vs:6.2f} {rho:6.2f} {h_str:>8}")

print(
    f"\nSlownesses: {len(cfg.p_values)} values, "
    f"p = {cfg.p_values[0]:.2f} -- {cfg.p_values[-1]:.2f} s/km"
)
print(f"Frequencies: {cfg.nfreq}")
print(f"Perturbation: {cfg.perturbation:.0%}")
print(f"Max iterations: {cfg.max_iter}")

## Run Inversion

In [ ]:
from GlobalMatrix.taup_inversion import invert_taup

result = invert_taup(
    true_model=cfg.true_model,
    p_values=cfg.p_values,
    nfreq=cfg.nfreq,
    perturbation=cfg.perturbation,
    max_iter=cfg.max_iter,
    seed=cfg.seed,
    tol=cfg.tol,
)

print(f"\nConverged: {result.converged}")
print(f"Iterations: {result.n_iterations}")
print(f"Final misfit: {result.misfit_history[-1]:.2e}")
print(f"Final param error: {result.param_error_history[-1]:.4e}")

## Convergence Curves

In [ ]:
from Kennett_Reflectivity.taup_inversion import _LATEX_RCPARAMS

iters = np.arange(1, len(result.misfit_history) + 1)

with plt.rc_context(_LATEX_RCPARAMS):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].semilogy(iters, result.misfit_history, "b-o", markersize=3)
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel(r"Misfit $\chi$")
    axes[0].set_title("Misfit convergence")
    axes[0].grid(True, alpha=0.3)

    axes[1].semilogy(iters, result.grad_norm_history, "r-o", markersize=3)
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel(r"$\|\nabla\chi\|$")
    axes[1].set_title("Gradient norm")
    axes[1].grid(True, alpha=0.3)

    axes[2].semilogy(iters, result.param_error_history, "k-o", markersize=3)
    axes[2].set_xlabel("Iteration")
    axes[2].set_ylabel(
        r"$\|\mathbf{m} - \mathbf{m}_\mathrm{true}\|"
        r" / \|\mathbf{m}_\mathrm{true}\|$"
    )
    axes[2].set_title("Relative parameter error")
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## Model Comparison

In [ ]:
header = (
    f"{'Layer':<15} {'Param':>6} {'True':>8} {'Initial':>8} "
    f"{'Recovered':>10} {'Error %':>8}"
)
print(header)
print("-" * len(header))

for i in range(cfg.true_model.n_layers):
    name = cfg.layer_names[i]
    for param, label in [
        ("alpha", "Vp"),
        ("beta", "Vs"),
        ("rho", "rho"),
        ("thickness", "h"),
    ]:
        t_val = getattr(result.true_model, param)[i]
        i_val = getattr(result.initial_model, param)[i]
        r_val = getattr(result.recovered_model, param)[i]

        if np.isinf(t_val):
            print(f"{name:<15} {label:>6} {'inf':>8} {'inf':>8} {'inf':>10} {'---':>8}")
        else:
            err = abs(r_val - t_val) / max(abs(t_val), 1e-30) * 100
            print(
                f"{name:<15} {label:>6} {t_val:8.4f} {i_val:8.4f} "
                f"{r_val:10.4f} {err:8.2e}"
            )
        name = ""  # only print layer name on first row

## Depth Profiles

In [ ]:
from Kennett_Reflectivity.taup_inversion import _depth_profile

params = [
    ("alpha", r"$V_P$ (km/s)"),
    ("beta", r"$V_S$ (km/s)"),
    ("rho", r"$\rho$ (g/cm$^3$)"),
]
models = [
    (result.true_model, "True", "black", "-", 1.5),
    (result.initial_model, "Initial", "#2980b9", "--", 1.0),
    (result.recovered_model, "Recovered", "#c0392b", "-", 1.2),
]

with plt.rc_context(_LATEX_RCPARAMS):
    fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)

    for ax, (param, xlabel) in zip(axes, params):
        for model, label, color, ls, lw in models:
            depths, values = _depth_profile(model, param)
            ax.plot(
                values, depths, color=color, linestyle=ls, linewidth=lw, label=label
            )
        ax.set_xlabel(xlabel)
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("Depth (km)")
    axes[0].legend(loc="lower left", fontsize=9)
    fig.tight_layout()
    plt.show()

## Tau-p Trace Comparison

In [ ]:
from GlobalMatrix.taup_inversion import compute_taup_traces

td = cfg.output.trace_display
p_plot = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]

_, traces_true = compute_taup_traces(result.true_model, p_plot, nw=td.nw)
_, traces_rec = compute_taup_traces(result.recovered_model, p_plot, nw=td.nw)
time = np.arange(2 * td.nw, dtype=np.float64) * (64.0 / (2 * td.nw))

tmask = time <= td.t_max
t_plot = time[tmask]
global_peak = max(np.abs(traces_true[p][tmask]).max() for p in p_plot)
dp = p_plot[1] - p_plot[0]
clip = 0.8

with plt.rc_context(_LATEX_RCPARAMS):
    fig, (ax_main, ax_res) = plt.subplots(1, 2, figsize=(14, 10), sharey=True)

    for p_val in p_plot:
        tr_t = traces_true[p_val][tmask].copy()
        tr_r = traces_rec[p_val][tmask].copy()
        if global_peak > 0:
            tr_t = tr_t / global_peak * dp * clip
            tr_r = tr_r / global_peak * dp * clip

        ax_main.plot(p_val + tr_t, t_plot, "k-", linewidth=0.4)
        ax_main.fill_betweenx(
            t_plot,
            p_val,
            p_val + tr_t,
            where=(tr_t > 0),
            interpolate=True,
            color="black",
            alpha=0.9,
        )
        ax_main.plot(p_val + tr_r, t_plot, color="#c0392b", linewidth=0.8)

    ax_main.set_xlabel(r"Slowness $p$ (s/km)")
    ax_main.set_ylabel("Time (s)")
    ax_main.set_title("True (black) vs Recovered (red)")
    ax_main.invert_yaxis()
    ax_main.set_xlim(p_plot[0] - dp, p_plot[-1] + dp)

    # Residual
    res_peak = max(
        np.abs(traces_true[p][tmask] - traces_rec[p][tmask]).max() for p in p_plot
    )
    res_scale = global_peak if res_peak == 0 else res_peak

    for p_val in p_plot:
        residual = traces_true[p_val][tmask] - traces_rec[p_val][tmask]
        if res_scale > 0:
            residual = residual / res_scale * dp * clip
        ax_res.plot(p_val + residual, t_plot, color="#2980b9", linewidth=0.5)
        ax_res.fill_betweenx(
            t_plot,
            p_val,
            p_val + residual,
            where=(residual > 0),
            interpolate=True,
            color="#2980b9",
            alpha=0.6,
        )

    if global_peak > 0 and res_peak > 0:
        ratio = res_peak / global_peak
        ax_res.set_title(rf"Residual (peak $= {ratio:.1e} \times$ signal)")
    else:
        ax_res.set_title("Residual")
    ax_res.set_xlabel(r"Slowness $p$ (s/km)")
    ax_res.set_xlim(p_plot[0] - dp, p_plot[-1] + dp)

    fig.tight_layout()
    plt.show()

## Export Outputs

In [ ]:
from Kennett_Reflectivity.inversion_config import save_config
from Kennett_Reflectivity.taup_inversion import (
    plot_convergence_curves,
    write_model_profiles_tikz,
    write_model_table_latex,
)

from GlobalMatrix.taup_inversion import plot_taup_traces

outdir = cfg.output.directory
outdir.mkdir(parents=True, exist_ok=True)

# LaTeX model parameter table
write_model_table_latex(
    result,
    outdir / "gmm_taup_model_parameters.tex",
    layer_names=cfg.layer_names,
)
print(f"LaTeX table  -> {outdir / 'gmm_taup_model_parameters.tex'}")

# TikZ depth profiles
write_model_profiles_tikz(result, outdir / "gmm_taup_model_profiles.tex")
print(f"Depth profiles -> {outdir / 'gmm_taup_model_profiles.tex'}")

# Trace comparison PDF
td = cfg.output.trace_display
plot_taup_traces(
    result,
    outdir / "gmm_taup_trace_comparison.pdf",
    nw=td.nw,
    t_max=td.t_max,
)
print(f"Trace comparison -> {outdir / 'gmm_taup_trace_comparison.pdf'}")

# Convergence PDF
plot_convergence_curves(result, outdir / "gmm_taup_convergence.pdf")
print(f"Convergence -> {outdir / 'gmm_taup_convergence.pdf'}")

# Config YAML for reproducibility
save_config(cfg, outdir / "gmm_inversion_config.yaml")
print(f"Config saved -> {outdir / 'gmm_inversion_config.yaml'}")